[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/03-index-configuration/02-managing_schemas_with_tableconfig.ipynb)

# Managing Schemas with TableConfig

The previous notebook built a `TableConfig` in one shot: define every `TableFieldConfig` up front, pass the full list into `TableConfig(fields=[...])`, done. That works for a schema you write once and never touch again.

Real schemas change. A new column gets added to your data source. A field that used to be `TERM` needs to become `PHRASE`. A payload column turns out not to be needed at all. `TableConfig` has dedicated methods for exactly this: adding, looking up, updating, removing, and validating fields, without having to rebuild the whole schema from scratch every time.

In this notebook you will:

1. Start from an empty `TableConfig` and build it up field by field
2. Look up an existing field's configuration
3. Update a field's configuration after the fact
4. Remove a field entirely
5. Validate that a schema has no duplicate or conflicting field names

In [ ]:
# !pip install mbox

## 1. Building a schema incrementally

Instead of writing out a full `fields=[...]` list, start with an empty `TableConfig` and add fields one at a time with `add_field()`.

In [1]:
from mbox.config import TableConfig, TableFieldConfig, IndexType

schema = TableConfig(fields=[])

schema.add_field(TableFieldConfig(column="product_id", index_type=IndexType.IDENT))
schema.add_field(TableFieldConfig(column="product_name", index_type=IndexType.PHRASE))
schema.add_field(TableFieldConfig(column="unit_price", index_type=IndexType.DOUBLE))

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


True

`add_field()` returns `True` on success. If you try to add a field for a column that's already in the schema, it returns `False` instead of raising an error or silently overwriting the existing definition, so it's safe to check.

In [2]:
# Attempting to add "product_id" again does nothing, and tells you so
duplicate_attempt = schema.add_field(
    TableFieldConfig(column="product_id", index_type=IndexType.TERM)
)

print("Added successfully:", duplicate_attempt)

Added successfully: False


This matters in practice whenever a schema is being built up from more than one source, for example, a base schema plus columns discovered dynamically from a DataFrame. `add_field()` protects you from accidentally clobbering a field you already configured on purpose.

## 2. Looking up a field's configuration

`get_field()` retrieves the current `TableFieldConfig` for a given column name, or `None` if it isn't in the schema.

In [3]:
product_id_config = schema.get_field("product_id")
print(product_id_config)

missing = schema.get_field("does_not_exist")
print(missing)

column='product_id' index_type=<IndexType.IDENT: 'Generic - Identity'> character_mapping=None aliases=None
None


This is useful when a schema was built somewhere else, perhaps loaded from JSON, discussed in the next notebook, and you want to inspect or reuse one field's settings without printing the whole thing.

## 3. Updating a field after the fact

Suppose you decide `product_name` should be treated as `TERM` instead of `PHRASE`, perhaps you've learned your product names are consistently single, distinctive tokens rather than full phrases. `update_field()` replaces an existing field's configuration in place.

In [4]:
updated = schema.update_field(
    field=TableFieldConfig(column="product_name", index_type=IndexType.TERM),
    column="product_name"
)

print("Updated successfully:", updated)
print(schema.get_field("product_name"))

Updated successfully: True
column='product_name' index_type=<IndexType.TERM: 'Generic - Term'> character_mapping=None aliases=None


`update_field()` returns `False` if the column you're trying to update doesn't exist yet, mirroring `add_field()`'s behavior for duplicates. If you want "add this field, or update it if it's already there," check the return value and fall back to `add_field()` when `update_field()` returns `False`.

## 4. Removing a field

`remove_field()` takes a column name out of the schema entirely.

In [5]:
removed = schema.remove_field("unit_price")
print("Removed successfully:", removed)
print(schema.get_field("unit_price"))

Removed successfully: True
None


Once removed, `get_field()` correctly returns `None`. If `unit_price` needs to come back later, it has to be re-added with `add_field()`, not restored from anywhere, `TableConfig` doesn't keep a history of removed fields.

## 5. Validating a schema

`validate_fields()` checks that every column name in the schema is unique. This matters most once a schema is being assembled from multiple sources, hand-written fields merged with fields loaded from a file, for instance, where a silent duplicate could otherwise slip in unnoticed.

In [6]:
print("Schema is valid:", schema.validate_fields())

Schema is valid: True


Under normal use, `add_field()` and `update_field()` already prevent duplicates from being introduced through those methods. `validate_fields()` is most useful as a final sanity check, for example, right before passing a schema into `TableIndexer.create_index()`, especially if the schema's `fields` list was ever constructed or modified by hand rather than exclusively through the methods above.

## 6. Building an index from the finished schema

Once the schema looks right, use it exactly as before, via `config_overrides`.

In [7]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.DataFrame({
    "product_id": ["B88-EXT", "A12-PWR", "C99-SNS"],
    "product_name": ["Extended Battery Pack", "Portable Power Bank", "Motion Sensor Camera"]
})

index = TableIndexer.create_index(
    df=df,
    config_overrides=schema,
    tmp_dir="tmp_index"
)

index.describe()

config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.


,Field,Index Type,Unique Value Count,Index size [bytes]
0,product_id,IndexType.IDENT,3,10428
1,product_name,IndexType.TERM,3,17972


Notice `unit_price` doesn't appear here at all, it was removed from the schema in Step 4, and the DataFrame we built for this section doesn't include it either. The schema and the data need to agree on which columns exist; `TableConfig`'s methods manage the schema side of that, but it's still your responsibility to keep the DataFrame in sync.

## Next steps

- **`03-schema_json_serialization_and_reuse.ipynb`** - save a schema like this one to JSON, and load it back in a different notebook, service, or pipeline run

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*